In [33]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, when, count
from pyspark.sql.types import DoubleType, FloatType

spark = SparkSession.builder \
    .appName("DataQualityChecks") \
    .config("spark.hadoop.io.native.lib.available", "false") \
    .getOrCreate()

In [23]:
df = spark.read.csv(
    r"C:\Water_Monitoring_System\data\brisbane_water_quality.csv",
    header=True,
    inferSchema=True
)

print("Sample Data:")
df.show(10)

Sample Data:
+-------------------+-------------+-------------------+-----------------------+-----------+---------------------+-----------+---------------------+----------------+--------------------------+------------------------------+----------------------------------------+-----+------------+--------+------------------+--------------------+------------------------------+---------+-------------------+
|          Timestamp|Record number|Average Water Speed|Average Water Direction|Chlorophyll|Chlorophyll [quality]|Temperature|Temperature [quality]|Dissolved Oxygen|Dissolved Oxygen [quality]|Dissolved Oxygen (%Saturation)|Dissolved Oxygen (%Saturation) [quality]|   pH|pH [quality]|Salinity|Salinity [quality]|Specific Conductance|Specific Conductance [quality]|Turbidity|Turbidity [quality]|
+-------------------+-------------+-------------------+-----------------------+-----------+---------------------+-----------+---------------------+----------------+--------------------------+----------

In [25]:
print("Missing Values:")

missing_exprs = []

for field in df.schema.fields:
    column_name = field.name
    data_type = field.dataType

    # Numeric columns 
    if isinstance(data_type, (DoubleType, FloatType)):
        expr = count(
            when(col(column_name).isNull() | col(column_name).isNaN(), column_name)
        ).alias(column_name)
    
    # Other columns 
    else:
        expr = count(
            when(col(column_name).isNull(), column_name)
        ).alias(column_name)

    missing_exprs.append(expr)

df.select(missing_exprs).show()

Missing Values:
+---------+-------------+-------------------+-----------------------+-----------+---------------------+-----------+---------------------+----------------+--------------------------+------------------------------+----------------------------------------+----+------------+--------+------------------+--------------------+------------------------------+---------+-------------------+
|Timestamp|Record number|Average Water Speed|Average Water Direction|Chlorophyll|Chlorophyll [quality]|Temperature|Temperature [quality]|Dissolved Oxygen|Dissolved Oxygen [quality]|Dissolved Oxygen (%Saturation)|Dissolved Oxygen (%Saturation) [quality]|  pH|pH [quality]|Salinity|Salinity [quality]|Specific Conductance|Specific Conductance [quality]|Turbidity|Turbidity [quality]|
+---------+-------------+-------------------+-----------------------+-----------+---------------------+-----------+---------------------+----------------+--------------------------+------------------------------+--------

In [27]:
# Range Validation
print("Invalid Data:")

invalid = df.filter(
    (col("pH") < 0) | (col("pH") > 14) |
    (col("Dissolved Oxygen") < 0) | (col("Dissolved Oxygen") > 20)
)

invalid.show()

Invalid Data:
+---------+-------------+-------------------+-----------------------+-----------+---------------------+-----------+---------------------+----------------+--------------------------+------------------------------+----------------------------------------+---+------------+--------+------------------+--------------------+------------------------------+---------+-------------------+
|Timestamp|Record number|Average Water Speed|Average Water Direction|Chlorophyll|Chlorophyll [quality]|Temperature|Temperature [quality]|Dissolved Oxygen|Dissolved Oxygen [quality]|Dissolved Oxygen (%Saturation)|Dissolved Oxygen (%Saturation) [quality]| pH|pH [quality]|Salinity|Salinity [quality]|Specific Conductance|Specific Conductance [quality]|Turbidity|Turbidity [quality]|
+---------+-------------+-------------------+-----------------------+-----------+---------------------+-----------+---------------------+----------------+--------------------------+------------------------------+------------

In [37]:
 # Anomaly Detection

print("Anomalies:")

anomalies = df.filter(
    (col("pH") < 6) | (col("pH") > 8.5) |
    (col("Dissolved Oxygen") < 4) |
    (col("Turbidity") > 30)
)

anomalies.show()

Anomalies:
+-------------------+-------------+-------------------+-----------------------+-----------+---------------------+-----------+---------------------+----------------+--------------------------+------------------------------+----------------------------------------+-----+------------+--------+------------------+--------------------+------------------------------+---------+-------------------+
|          Timestamp|Record number|Average Water Speed|Average Water Direction|Chlorophyll|Chlorophyll [quality]|Temperature|Temperature [quality]|Dissolved Oxygen|Dissolved Oxygen [quality]|Dissolved Oxygen (%Saturation)|Dissolved Oxygen (%Saturation) [quality]|   pH|pH [quality]|Salinity|Salinity [quality]|Specific Conductance|Specific Conductance [quality]|Turbidity|Turbidity [quality]|
+-------------------+-------------+-------------------+-----------------------+-----------+---------------------+-----------+---------------------+----------------+--------------------------+------------

In [39]:
clean_df = df.filter(
    (col("pH") >= 0) & (col("pH") <= 14) &
    (col("Dissolved Oxygen") >= 0) & (col("Dissolved Oxygen") <= 20)
)

print("Clean Data:")
clean_df.show(10)

Clean Data:
+-------------------+-------------+-------------------+-----------------------+-----------+---------------------+-----------+---------------------+----------------+--------------------------+------------------------------+----------------------------------------+-----+------------+--------+------------------+--------------------+------------------------------+---------+-------------------+
|          Timestamp|Record number|Average Water Speed|Average Water Direction|Chlorophyll|Chlorophyll [quality]|Temperature|Temperature [quality]|Dissolved Oxygen|Dissolved Oxygen [quality]|Dissolved Oxygen (%Saturation)|Dissolved Oxygen (%Saturation) [quality]|   pH|pH [quality]|Salinity|Salinity [quality]|Specific Conductance|Specific Conductance [quality]|Turbidity|Turbidity [quality]|
+-------------------+-------------+-------------------+-----------------------+-----------+---------------------+-----------+---------------------+----------------+--------------------------+-----------